# Chapter 8. Hierarchical Model

- In Sect. 8.1, we introduce the motivation of building a hierarchical model. For 
a dataset example, we implement different models, from a simple model to a 
hierarchical model, and discuss several features of hierarchical models. 
- In Sect. 8.2, we expand the example in Sect. 8.1, and introduce how to build a 
hierarchical model that has multiple levels. 
- In Sects. 8.3 and 8.5, we learn how to apply hierarchical models to other datasets. In 
particular, we build a hierarchical model based on the nonlinear model mentioned 
in Sect. 7.2 and a logistic regression model mentioned in Sect. 5.4. 
- In Sect. 8.4, we introduce how to handle data that include missing values. Data 
transformation from “wide-format” to “long-format” is key

# 8.1 Intro to Hierarchical Models

- X: length of work experience of each employee (X, unit: years). 40 people in total.
- Y : employee’s annual income (Y , unit: $1 k). 
- CID: Company ID, that is, index of the company that the employee belongs to. There are 4 companies in total.

In [5]:
import pandas as pd
from cmdstanpy import CmdStanModel

# Read the CSV data
df = pd.read_csv('input/data-salary-2.csv')
print(df.describe())
df.tail()

               X          Y        CID
count  40.000000  40.000000  40.000000
mean   15.550000  54.795000   2.025000
std     7.815009  10.841302   0.973692
min     1.000000  37.600000   1.000000
25%     9.750000  45.600000   1.000000
50%    17.000000  53.200000   2.000000
75%    22.000000  62.750000   3.000000
max    28.000000  76.300000   4.000000


,X,Y,CID
35,22,64.1,3
36,25,59.2,3
37,28,72.2,4
38,24,72.6,4
39,22,72.8,4


In [26]:
stan_file = 'model/model8-1.stan'
with open(stan_file, 'r') as f:
    print(f.read())

data {
  int N;
  vector[N] X;
  vector[N] Y;
}

parameters {
  real a;
  real b;
  real<lower=0> s_y;
}

model {
  Y[1:N] ~ normal(a + b*X[1:N], s_y);
}



In [32]:
# Prepare the Stan model
stan_model = CmdStanModel(stan_file=stan_file)
fit = stan_model.sample(data=stan_data)

14:30:06 - cmdstanpy - INFO - CmdStan start processing
chain 1 |          | 00:00 Status


chain 1 |██████████| 00:00 Sampling completed
chain 2 |██████████| 00:00 Sampling completed
chain 3 |██████████| 00:00 Sampling completed
chain 4 |██████████| 00:00 Sampling completed


14:30:06 - cmdstanpy - INFO - CmdStan done processing.


In [45]:
quantile_pairs = samples_df[['a', 'b', 's_y']].quantile([0.025, 0.975])
print(quantile_pairs)
for param in ['a', 'b', 's_y']:
    print(f"{param}: mean: {samples_df[param].mean():.2f}, 95% CI: [{quantile_pairs.loc[0.025, param]:.2f}, {quantile_pairs.loc[0.975, param]:.2f}]")

               a         b       s_y
0.025  32.624705  0.830763  5.463993
0.975  42.372145  1.385567  8.520984
a: mean: 37.60, 95% CI: [32.62, 42.37]
b: mean: 1.11, 95% CI: [0.83, 1.39]
s_y: mean: 6.84, 95% CI: [5.46, 8.52]


---

In [46]:
stan_file = 'model/model8-2.stan'
with open(stan_file, 'r') as f:
    print(f.read())

data {
  int N;
  int C;
  vector[N] X;
  vector[N] Y;
  array[N] int<lower=1, upper=C> n2c;
}

parameters {
  vector[C] a;
  vector[C] b;
  real<lower=0> s_y;
}

model {
  Y[1:N] ~ normal(a[n2c] + b[n2c] .* X[1:N], s_y);
}



In [50]:
# Prepare the Stan model
stan_model = CmdStanModel(stan_file=stan_file)
fit = stan_model.sample(data=stan_data)
samples_df = fit.draws_pd()

14:42:10 - cmdstanpy - INFO - CmdStan start processing
chain 1 |          | 00:00 Status


chain 1 |████      | 00:00 Iteration:  700 / 2000 [ 35%]  (Warmup)


chain 1 |█████████▌| 00:00 Iteration: 1800 / 2000 [ 90%]  (Sampling)


chain 1 |██████████| 00:00 Sampling completed                       
chain 2 |██████████| 00:00 Sampling completed                       
chain 3 |██████████| 00:00 Sampling completed                       
chain 4 |██████████| 00:00 Sampling completed                       


14:42:10 - cmdstanpy - INFO - CmdStan done processing.


In [52]:
quantile_pairs = samples_df[['a[1]', 'a[2]', 'a[3]', 'a[4]', 'b[1]', 'b[2]', 'b[3]', 'b[4]', 's_y']].quantile([0.025, 0.975])
# print(quantile_pairs)
for param in ['a[1]', 'a[2]', 'a[3]', 'a[4]', 'b[1]', 'b[2]', 'b[3]', 'b[4]', 's_y']:
    mean_val = samples_df[param].mean()
    ci_low = quantile_pairs.loc[0.025, param]
    ci_high = quantile_pairs.loc[0.975, param]
    print(f"{param}: mean: {mean_val:.2f}, 95% CI: [{ci_low:.2f}, {ci_high:.2f}]")

a[1]: mean: 38.71, 95% CI: [36.07, 41.38]
a[2]: mean: 32.87, 95% CI: [29.65, 36.14]
a[3]: mean: 31.48, 95% CI: [24.68, 38.25]
a[4]: mean: 74.79, 95% CI: [41.13, 106.64]
b[1]: mean: 0.75, 95% CI: [0.58, 0.92]
b[2]: mean: 1.99, 95% CI: [1.75, 2.23]
b[3]: mean: 1.24, 95% CI: [0.91, 1.58]
b[4]: mean: -0.09, 95% CI: [-1.38, 1.25]
s_y: mean: 2.73, 95% CI: [2.12, 3.54]


---

In [54]:
# Hierarchical Model
stan_file = 'model/model8-3.stan'
with open(stan_file, 'r') as f:
    print(f.read())

data {
  int N;
  int C;
  vector[N] X;
  vector[N] Y;
  array[N] int<lower=1, upper=C> n2c;
}

parameters {
  real a_field;
  real b_field;
  vector[C] a_diff;
  vector[C] b_diff;
  real<lower=0> s_a;
  real<lower=0> s_b;
  real<lower=0> s_y;
}

transformed parameters {
  vector[C] a = a_field + a_diff[1:C];
  vector[C] b = b_field + b_diff[1:C];
}

model {
  a_diff[1:C] ~ normal(0, s_a);
  b_diff[1:C] ~ normal(0, s_b);
  Y[1:N] ~ normal(a[n2c] + b[n2c] .* X[1:N], s_y);
}



In [55]:
# Prepare the Stan model
stan_model = CmdStanModel(stan_file=stan_file)
fit = stan_model.sample(data=stan_data)
samples_df = fit.draws_pd()

14:44:28 - cmdstanpy - INFO - CmdStan start processing
chain 1 |          | 00:00 Status




chain 1 |▉         | 00:00 Iteration:    1 / 2000 [  0%]  (Warmup)
chain 1 |█▊        | 00:00 Iteration:  200 / 2000 [ 10%]  (Warmup)


chain 1 |███▏      | 00:00 Iteration:  500 / 2000 [ 25%]  (Warmup)


chain 1 |█████▉    | 00:00 Iteration: 1001 / 2000 [ 50%]  (Sampling)


chain 1 |███████▋  | 00:00 Iteration: 1400 / 2000 [ 70%]  (Sampling)


chain 1 |█████████▌| 00:00 Iteration: 1800 / 2000 [ 90%]  (Sampling)


chain 1 |██████████| 00:00 Sampling completed                       
chain 2 |██████████| 00:00 Sampling completed                       
chain 3 |██████████| 00:00 Sampling completed                       
chain 4 |██████████| 00:00 Sampling completed                       


14:44:29 - cmdstanpy - INFO - CmdStan done processing.
14:44:29 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 1 iterations at max treedepth (0.1%)
	Chain 2 had 1 divergent transitions (0.1%)
	Chain 2 had 1 iterations at max treedepth (0.1%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


In [59]:
quantile_pairs = samples_df[['a_field', 'b_field', 's_a', 's_b', 's_y']].quantile([0.025, 0.975])
for param in ['a_field', 'b_field', 's_a', 's_b', 's_y']:
    mean_val = samples_df[param].mean()
    ci_low = quantile_pairs.loc[0.025, param]
    ci_high = quantile_pairs.loc[0.975, param]
    print(f"{param}: mean: {mean_val:.2f}, 95% CI: [{ci_low:.2f}, {ci_high:.2f}]")

a_field: mean: 38.65, 95% CI: [21.26, 65.47]
b_field: mean: 1.24, 95% CI: [-0.47, 2.90]
s_a: mean: 14.77, 95% CI: [1.22, 60.07]
s_b: mean: 1.19, 95% CI: [0.31, 4.40]
s_y: mean: 2.87, 95% CI: [2.20, 3.73]


In [ ]:
quantile_pairs = samples_df[['a_field', 'b_field', 's_a', 's_b', 's_y']].quantile([0.025, 0.975])
for param in ['a_field', 'b_field', 's_a', 's_b', 's_y']:
    mean_val = samples_df[param].median()
    ci_low = quantile_pairs.loc[0.025, param]
    ci_high = quantile_pairs.loc[0.975, param]
    print(f"{param}: median: {mean_val:.1f}, 95% CI: [{ci_low:.1f}, {ci_high:.1f}]")

a_field: median: 36.7, 95% CI: [13.9, 60.7]
b_field: median: 1.2, 95% CI: [-0.4, 2.8]
s_a: median: 9.6, 95% CI: [1.4, 76.4]
s_b: median: 0.8, 95% CI: [0.3, 4.6]
s_y: median: 2.8, 95% CI: [2.2, 3.8]


---

In [60]:
# Hierarchical Model
stan_file = 'model/model8-4.stan'
with open(stan_file, 'r') as f:
    print(f.read())

data {
  int N;
  int C;
  vector[N] X;
  vector[N] Y;
  array[N] int<lower=1, upper=C> n2c;
}

parameters {
  real a_field;
  real b_field;
  vector[C] a;
  vector[C] b;
  real<lower=0> s_a;
  real<lower=0> s_b;
  real<lower=0> s_y;
}

model {
  a[1:C] ~ normal(a_field, s_a);
  b[1:C] ~ normal(b_field, s_b);
  Y[1:N] ~ normal(a[n2c] + b[n2c] .* X[1:N], s_y);
}



In [61]:
# Prepare the Stan model
stan_model = CmdStanModel(stan_file=stan_file)
fit = stan_model.sample(data=stan_data)
samples_df = fit.draws_pd()

14:48:53 - cmdstanpy - INFO - CmdStan start processing
chain 1 |          | 00:00 Status



chain 1 |██▋       | 00:00 Iteration:  400 / 2000 [ 20%]  (Warmup)


chain 1 |████████▏ | 00:00 Iteration: 1500 / 2000 [ 75%]  (Sampling)

chain 1 |██████████| 00:00 Sampling completed                       
chain 2 |██████████| 00:00 Sampling completed                       
chain 3 |██████████| 00:00 Sampling completed                       
chain 4 |██████████| 00:00 Sampling completed                       


14:48:53 - cmdstanpy - INFO - CmdStan done processing.
14:48:53 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 6 divergent transitions (0.6%)
	Chain 2 had 2 divergent transitions (0.2%)
	Chain 4 had 23 divergent transitions (2.3%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


In [65]:
quantile_pairs = samples_df[['a_field', 'b_field', 's_a', 's_b', 's_y']].quantile([0.025, 0.975])
for param in ['a_field', 'b_field', 's_a', 's_b', 's_y']:
    mean_val = samples_df[param].median()
    ci_low = quantile_pairs.loc[0.025, param]
    ci_high = quantile_pairs.loc[0.975, param]
    print(f"{param}: median: {mean_val:.1f}, 95% CI: [{ci_low:.1f}, {ci_high:.1f}]")

a_field: median: 36.7, 95% CI: [13.9, 60.7]
b_field: median: 1.2, 95% CI: [-0.4, 2.8]
s_a: median: 9.6, 95% CI: [1.4, 76.4]
s_b: median: 0.8, 95% CI: [0.3, 4.6]
s_y: median: 2.8, 95% CI: [2.2, 3.8]


In [63]:
quantile_pairs = samples_df[['a[1]', 'a[2]', 'a[3]', 'a[4]', 'b[1]', 'b[2]', 'b[3]', 'b[4]', 's_y']].quantile([0.025, 0.975])
# print(quantile_pairs)
for param in ['a[1]', 'a[2]', 'a[3]', 'a[4]', 'b[1]', 'b[2]', 'b[3]', 'b[4]', 's_y']:
    mean_val = samples_df[param].mean()
    ci_low = quantile_pairs.loc[0.025, param]
    ci_high = quantile_pairs.loc[0.975, param]
    print(f"{param}: mean: {mean_val:.2f}, 95% CI: [{ci_low:.2f}, {ci_high:.2f}]")

a[1]: mean: 38.37, 95% CI: [35.15, 41.37]
a[2]: mean: 33.50, 95% CI: [30.12, 36.95]
a[3]: mean: 32.46, 95% CI: [25.45, 38.88]
a[4]: mean: 47.97, 95% CI: [30.10, 80.83]
b[1]: mean: 0.77, 95% CI: [0.59, 0.96]
b[2]: mean: 1.94, 95% CI: [1.68, 2.19]
b[3]: mean: 1.19, 95% CI: [0.88, 1.54]
b[4]: mean: 0.99, 95% CI: [-0.34, 1.71]
s_y: mean: 2.86, 95% CI: [2.21, 3.76]


---

Provide some proper priors to avoid divergent chain

In [ ]:
# Hierarchical Model
stan_file = 'model/model8-4.stan'
with open(stan_file, 'r') as f:
    print(f.read())

data {
  int N;
  int C;
  vector[N] X;
  vector[N] Y;
  array[N] int<lower=1, upper=C> n2c;
}

parameters {
  real a_field;
  real b_field;
  vector[C] a;
  vector[C] b;
  real<lower=0> s_a;
  real<lower=0> s_b;
  real<lower=0> s_y;
}

model {
  a[1:C] ~ normal(a_field, s_a);
  b[1:C] ~ normal(b_field, s_b);
  Y[1:N] ~ normal(a[n2c] + b[n2c] .* X[1:N], s_y);
}



In [71]:
# Prepare the Stan model
stan_model = CmdStanModel(stan_file=stan_file)
fit = stan_model.sample(data=stan_data, adapt_delta=0.999)
samples_df = fit.draws_pd()

14:54:50 - cmdstanpy - INFO - CmdStan start processing
chain 1 |          | 00:00 Status



chain 1 |▉         | 00:00 Iteration:    1 / 2000 [  0%]  (Warmup)


chain 1 |██▋       | 00:00 Iteration:  400 / 2000 [ 20%]  (Warmup)


chain 1 |████▌     | 00:00 Iteration:  800 / 2000 [ 40%]  (Warmup)


chain 1 |██████▎   | 00:00 Iteration: 1100 / 2000 [ 55%]  (Sampling)


chain 1 |███████▋  | 00:00 Iteration: 1400 / 2000 [ 70%]  (Sampling)

chain 1 |█████████ | 00:00 Iteration: 1700 / 2000 [ 85%]  (Sampling)


chain 1 |██████████| 00:02 Sampling completed                       
chain 2 |██████████| 00:02 Sampling completed                       
chain 3 |██████████| 00:01 Sampling completed                       
chain 4 |██████████| 00:01 Sampling completed                       


14:54:52 - cmdstanpy - INFO - CmdStan done processing.
14:54:52 - cmdstanpy - WARNING - Some chains may have failed to converge.
	Chain 1 had 2 divergent transitions (0.2%)
	Chain 2 had 407 iterations at max treedepth (40.7%)
	Use the "diagnose()" method on the CmdStanMCMC object to see further information.


In [70]:
print(fit.diagnose())

Checking sampler transitions treedepth.
Treedepth satisfactory for all transitions.

Checking sampler transitions for divergences.
7 of 1000 (0.70%) transitions ended with a divergence.
These divergent transitions indicate that HMC is not fully able to explore the posterior distribution.
Try increasing adapt delta closer to 1.
If this doesn't remove all divergences, try to reparameterize the model.

Checking E-BFMI - sampler transitions HMC potential energy.
E-BFMI satisfactory.

Rank-normalized split effective sample size satisfactory for all parameters.

Rank-normalized split R-hat values satisfactory for all parameters.

Processing complete.



In [ ]:
quantile_pairs = samples_df[['a_field', 'b_field', 's_a', 's_b', 's_y']].quantile([0.025, 0.975])
for param in ['a_field', 'b_field', 's_a', 's_b', 's_y']:
    mean_val = samples_df[param].median()
    ci_low = quantile_pairs.loc[0.025, param]
    ci_high = quantile_pairs.loc[0.975, param]
    print(f"{param}: median: {mean_val:.1f}, 95% CI: [{ci_low:.1f}, {ci_high:.1f}]")

a_field: median: 36.7, 95% CI: [13.9, 60.7]
b_field: median: 1.2, 95% CI: [-0.4, 2.8]
s_a: median: 9.6, 95% CI: [1.4, 76.4]
s_b: median: 0.8, 95% CI: [0.3, 4.6]
s_y: median: 2.8, 95% CI: [2.2, 3.8]


In [ ]:
quantile_pairs = samples_df[['a[1]', 'a[2]', 'a[3]', 'a[4]', 'b[1]', 'b[2]', 'b[3]', 'b[4]', 's_y']].quantile([0.025, 0.975])
# print(quantile_pairs)
for param in ['a[1]', 'a[2]', 'a[3]', 'a[4]', 'b[1]', 'b[2]', 'b[3]', 'b[4]', 's_y']:
    mean_val = samples_df[param].mean()
    ci_low = quantile_pairs.loc[0.025, param]
    ci_high = quantile_pairs.loc[0.975, param]
    print(f"{param}: mean: {mean_val:.2f}, 95% CI: [{ci_low:.2f}, {ci_high:.2f}]")

a[1]: mean: 38.37, 95% CI: [35.15, 41.37]
a[2]: mean: 33.50, 95% CI: [30.12, 36.95]
a[3]: mean: 32.46, 95% CI: [25.45, 38.88]
a[4]: mean: 47.97, 95% CI: [30.10, 80.83]
b[1]: mean: 0.77, 95% CI: [0.59, 0.96]
b[2]: mean: 1.94, 95% CI: [1.68, 2.19]
b[3]: mean: 1.19, 95% CI: [0.88, 1.54]
b[4]: mean: 0.99, 95% CI: [-0.34, 1.71]
s_y: mean: 2.86, 95% CI: [2.21, 3.76]


🧠 Why Is It Called "Shrinkage"?
In statistics, shrinkage refers to the phenomenon where estimates are pulled toward a central value—typically the group mean or overall average. This is especially prominent in Bayesian hierarchical models, which assume that individual parameters are drawn from a shared distribution.

In non-hierarchical models, each group is estimated independently, often resulting in high variability.

In hierarchical models, estimates are partially pooled, meaning they’re nudged toward the global mean.

So, the term "shrinkage" is appropriate—it reflects a reduction in variability, not necessarily in magnitude. It’s like saying: “Let’s not let each estimate go rogue; let’s gently guide them toward the center.”

🌟 Why Is Shrinkage So Important?
Shrinkage is a feature, not a flaw. It plays a critical role in improving model performance and interpretability:

Stabilizes estimates for small samples → Groups with few observations benefit from borrowing strength from others.

Reduces overfitting → By discouraging extreme estimates, shrinkage helps the model generalize better.

Improves predictive accuracy → Conservative estimates often perform better on new, unseen data.

Reflects uncertainty → Shrinkage acknowledges that we don’t know everything about each group, so we temper our confidence.

🔍 Related Concepts
Partial pooling: The mechanism behind shrinkage in hierarchical models.

James-Stein estimator: A classic example showing that shrinkage can outperform traditional estimators under certain conditions.

# Shrinkage in Bayesian Estimation

---

## 🧠 Why Is It Called "Shrinkage"?

In statistics, **shrinkage** refers to the phenomenon where estimates are pulled toward a central value—typically the group mean or overall average. This is especially prominent in **Bayesian hierarchical models**, which assume that individual parameters are drawn from a shared distribution.

- In **non-hierarchical models**, each group is estimated independently, often resulting in high variability.
- In **hierarchical models**, estimates are **partially pooled**, meaning they’re nudged toward the global mean.

So, the term "shrinkage" is appropriate—it reflects a **reduction in variability**, not necessarily in magnitude. It’s like saying: _“Let’s not let each estimate go rogue; let’s gently guide them toward the center.”_

---

## 🌟 Why Is Shrinkage So Important?

Shrinkage is a **feature**, not a flaw. It plays a critical role in improving model performance and interpretability:

- **Stabilizes estimates for small samples**  
  → Groups with few observations benefit from borrowing strength from others.

- **Reduces overfitting**  
  → By discouraging extreme estimates, shrinkage helps the model generalize better.

- **Improves predictive accuracy**  
  → Conservative estimates often perform better on new, unseen data.

- **Reflects uncertainty**  
  → Shrinkage acknowledges that we don’t know everything about each group, so we temper our confidence.

---

## 🔍 Related Concepts

- **Partial pooling**: The mechanism behind shrinkage in hierarchical models.
- **James-Stein estimator**: A classic example showing that shrinkage can outperform traditional estimators under certain conditions.

---

Yes, exactly! In Bayesian hierarchical models, **shrinkage** is often referred to as **partial pooling**.

---

## 🧩 What Is Partial Pooling?

**Partial pooling** is a modeling strategy where estimates for individual groups (e.g., schools, hospitals, players) are influenced by both their own data and the data from other groups. It’s the middle ground between:

- **No pooling**: Each group is estimated independently.
- **Complete pooling**: All groups are assumed to be the same and estimated together.

With **partial pooling**, each group’s estimate is “shrunk” toward the overall mean, but not forced to be identical. This is achieved by modeling group-level parameters as drawn from a shared distribution.

---

## 🔄 How It Relates to Shrinkage

- The **shrinkage effect** is the result of partial pooling.
- It helps stabilize estimates, especially for groups with limited data.
- It reflects the idea that groups are **similar but not identical**.

---

## 📌 Example

Imagine estimating batting averages for baseball players:

- A player with only 4 at-bats and no hits might look terrible in isolation.
- But with partial pooling, the model considers that this player might not be as bad as the raw data suggests—so their estimate is pulled toward the average of all players.

You can explore a great example of this in PyMC’s [Hierarchical Partial Pooling case study](https://www.pymc.io/projects/examples/en/latest/case_studies/hierarchical_partial_pooling.html).

---

Want to see how partial pooling compares visually to no pooling and complete pooling? I can sketch that out for you.